### Missing Indicator - Using pandas

In [1]:
import pandas as pd
import numpy as np

# ─── Inline dataset: Student exam data ───
data = {
    'Name':    ['Aarav', 'Priya', 'Ravi', 'Sneha', 'Karan', 'Divya', 'Mohit', 'Ananya'],
    'Age':     [22,      np.nan,  25,      23,      np.nan,  21,      24,      np.nan  ],
    'Score':   [85,      90,      np.nan,  78,      88,      np.nan,  72,      95      ],
    'Income':  [30000,   np.nan,  45000,   np.nan,  28000,   np.nan,  52000,   np.nan  ],
    'City':    ['Mumbai','Delhi', np.nan,  'Surat', 'Mumbai',np.nan,  'Delhi', 'Pune'  ],
}
df = pd.DataFrame(data)

print("Original Data:")
print(df)
print("\nMissing values:\n", df.isnull().sum())

# ─── Step 1: Indicator columns banao ───
# har column ke liye jahan NaN hai, 1 lagao, nahi toh 0
df['Age_Missing']    = df['Age'].isnull().astype(int)
df['Score_Missing']  = df['Score'].isnull().astype(int)
df['Income_Missing'] = df['Income'].isnull().astype(int)
df['City_Missing']   = df['City'].isnull().astype(int)

# ─── Step 2: Original columns fill karo ───
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Score'] = df['Score'].fillna(df['Score'].median())
df['Income'] = df['Income'].fillna(df['Income'].median())
df['City'] = df['City'].fillna(df['City'].mode()[0])

print("\nAfter Missing Indicator + Imputation:")
print(df[['Name', 'Age', 'Age_Missing', 'Income', 'Income_Missing']])
# Age_Missing = 1 wali rows mein imputed value hai
# Model seekhega: "Age_Missing=1 waale students ki pattern kaisi hai?"

Original Data:
     Name   Age  Score   Income    City
0   Aarav  22.0   85.0  30000.0  Mumbai
1   Priya   NaN   90.0      NaN   Delhi
2    Ravi  25.0    NaN  45000.0     NaN
3   Sneha  23.0   78.0      NaN   Surat
4   Karan   NaN   88.0  28000.0  Mumbai
5   Divya  21.0    NaN      NaN     NaN
6   Mohit  24.0   72.0  52000.0   Delhi
7  Ananya   NaN   95.0      NaN    Pune

Missing values:
 Name      0
Age       3
Score     2
Income    4
City      2
dtype: int64

After Missing Indicator + Imputation:
     Name   Age  Age_Missing   Income  Income_Missing
0   Aarav  22.0            0  30000.0               0
1   Priya  23.0            1  37500.0               1
2    Ravi  25.0            0  45000.0               0
3   Sneha  23.0            0  37500.0               1
4   Karan  23.0            1  28000.0               0
5   Divya  21.0            0  37500.0               1
6   Mohit  24.0            0  52000.0               0
7  Ananya  23.0            1  37500.0               1


### Missing Indicator - Using sklearn

In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, MissingIndicator

# ─── Same inline dataset ───
data = {
    'Age':    [22,  np.nan, 25,  23,  np.nan, 21,  24,  np.nan],
    'Score':  [85,  90,     np.nan, 78, 88,  np.nan, 72, 95   ],
    'Income': [30000, np.nan, 45000, np.nan, 28000, np.nan, 52000, np.nan],
}
df = pd.DataFrame(data)

# ─── MissingIndicator — sklearn ka official tool ───
indicator = MissingIndicator(features='missing-only')
# features='missing-only' → sirf unhi columns ke liye indicator banao jinmein NaN ho

indicator_array = indicator.fit_transform(df)
print("Indicator array shape:", indicator_array.shape)  # (8, 3) — 3 columns mein NaN tha

# Indicator DataFrame banao
indicator_cols = [df.columns[i] + '_Missing' for i in indicator.features_]
df_indicator = pd.DataFrame(indicator_array.astype(int), columns=indicator_cols)

# ─── Original columns impute karo ───
imputer = SimpleImputer(strategy='mean')
df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

# ─── Dono merge karo ───
df_final = pd.concat([df_imputed, df_indicator], axis=1)

print("\nFinal DataFrame with Indicators:")
print(df_final.round(1))
print("\nLearned missing column indices:", indicator.features_)
# [0, 1, 2] → sab teeno columns mein NaN tha

Indicator array shape: (8, 3)

Final DataFrame with Indicators:
    Age  Score   Income  Age_Missing  Score_Missing  Income_Missing
0  22.0   85.0  30000.0            0              0               0
1  23.0   90.0  38750.0            1              0               1
2  25.0   84.7  45000.0            0              1               0
3  23.0   78.0  38750.0            0              0               1
4  23.0   88.0  28000.0            1              0               0
5  21.0   84.7  38750.0            0              1               1
6  24.0   72.0  52000.0            0              0               0
7  23.0   95.0  38750.0            1              0               1

Learned missing column indices: [0 1 2]


### Random Sample Imputation

In [3]:
import pandas as pd
import numpy as np

# ─── Inline dataset: IPL Batting Stats ───
np.random.seed(42)
data = {
    'Player':       ['Rohit', 'Virat', 'Dhoni', 'Pant', 'Jadeja',
                     'Bumrah', 'Shami', 'KL Rahul', 'Hardik', 'Surya'],
    'Strike_Rate':  [130, np.nan, 135, 142, np.nan,
                     np.nan, 120, 145, np.nan, 170   ],
    'Avg_Score':    [38,  49,    np.nan, 32, 22,
                     12,  np.nan, 45,  35,  np.nan  ],
    'Matches':      [200, 215, 250, 100, 180,
                     120, np.nan, 105, 90, np.nan   ],
}
df = pd.DataFrame(data)

print("Original Data:")
print(df)
print("\nMissing values:\n", df.isnull().sum())

# ─── Random Sample Imputation ───

# Make a copy to work with, so the original DataFrame remains unchanged
df_imputed = df.copy()

# Set random seed for reproducibility
np.random.seed(42)

# --- Impute 'Strike_Rate' ---
if df_imputed['Strike_Rate'].isnull().sum() > 0:
    null_count_sr = df_imputed['Strike_Rate'].isnull().sum()    # 4
    random_values_sr = df_imputed['Strike_Rate'].dropna().sample(
        n=null_count_sr,
        replace=True,
        random_state=42
    ).values
    df_imputed.loc[df_imputed['Strike_Rate'].isnull(), 'Strike_Rate'] = random_values_sr

# --- Impute 'Avg_Score' ---
if df_imputed['Avg_Score'].isnull().sum() > 0:
    null_count_as = df_imputed['Avg_Score'].isnull().sum()
    random_values_as = df_imputed['Avg_Score'].dropna().sample(
        n=null_count_as,
        replace=True,
        random_state=42
    ).values
    df_imputed.loc[df_imputed['Avg_Score'].isnull(), 'Avg_Score'] = random_values_as

# --- Impute 'Matches' ---
if df_imputed['Matches'].isnull().sum() > 0:
    null_count_m = df_imputed['Matches'].isnull().sum()
    random_values_m = df_imputed['Matches'].dropna().sample(
        n=null_count_m,
        replace=True,
        random_state=42
    ).values
    df_imputed.loc[df_imputed['Matches'].isnull(), 'Matches'] = random_values_m

print("\nAfter Random Sample Imputation:")
print(df_imputed)

# ─── Distribution compare karo ───
print("\nStrike_Rate — Original Non-Null Stats:")
print(df['Strike_Rate'].dropna().describe().round(1))

print("\nStrike_Rate — After Imputation Stats:")
print(df_imputed['Strike_Rate'].describe().round(1))
# Mean aur std roughly same rahenge — distribution preserved!


Original Data:
     Player  Strike_Rate  Avg_Score  Matches
0     Rohit        130.0       38.0    200.0
1     Virat          NaN       49.0    215.0
2     Dhoni        135.0        NaN    250.0
3      Pant        142.0       32.0    100.0
4    Jadeja          NaN       22.0    180.0
5    Bumrah          NaN       12.0    120.0
6     Shami        120.0        NaN      NaN
7  KL Rahul        145.0       45.0    105.0
8    Hardik          NaN       35.0     90.0
9     Surya        170.0        NaN      NaN

Missing values:
 Player         0
Strike_Rate    4
Avg_Score      3
Matches        2
dtype: int64

After Random Sample Imputation:
     Player  Strike_Rate  Avg_Score  Matches
0     Rohit        130.0       38.0    200.0
1     Virat        120.0       49.0    215.0
2     Dhoni        135.0       35.0    250.0
3      Pant        142.0       32.0    100.0
4    Jadeja        145.0       22.0    180.0
5    Bumrah        142.0       12.0    120.0
6     Shami        120.0       22.0    105.

### KNN Imputer

In [4]:
import numpy as np
from sklearn.impute import KNNImputer

data = np.array([
    [2, 4, np.nan],
    [5, 1, 6],
    [np.nan, 5, 7],
    [9, 8, 9],
])

print("Original Data: ")
print(data)

imputer = KNNImputer(n_neighbors=2)

data_imputed = imputer.fit_transform(data)
print("Imputed Data:")
print(data_imputed)

Original Data: 
[[ 2.  4. nan]
 [ 5.  1.  6.]
 [nan  5.  7.]
 [ 9.  8.  9.]]
Imputed Data:
[[2.  4.  6.5]
 [5.  1.  6. ]
 [5.5 5.  7. ]
 [9.  8.  9. ]]


In [5]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# ─── Inline dataset: Swiggy Delivery Riders ───
data = {
    'Rider':           ['Raju', 'Suresh', 'Amit', 'Vikram', 'Pooja',
                        'Kiran', 'Deepak', 'Meena', 'Arjun', 'Sunita'],
    'Deliveries_Day':  [18,   25,      np.nan, 30,      15,
                        22,   28,      np.nan, 19,      26     ],
    'Exp_Years':       [3,    5,       4,      6,       np.nan,
                        4,    5,       3,      np.nan,  5      ],
    'Avg_Distance_km': [8.5,  12.0,    10.5,   np.nan,  7.0,
                        9.5,  np.nan,  8.0,    9.0,     11.5   ],
    'Rating':          [4.2,  4.7,     np.nan, 4.9,     4.0,
                        4.4,  4.8,     4.1,    np.nan,  4.6    ],
}
df = pd.DataFrame(data)
df_nums = df.drop('Rider', axis=1)   # sirf numerical columns

print("Original Data (numerical only):")
print(df_nums)
print("\nMissing values:\n", df_nums.isnull().sum())

# ─── KNN Imputer with k=3 ───
knn_imputer = KNNImputer(
    n_neighbors=3,           # 3 nearest neighbours use karo
    weights='distance'       # closer neighbours ka zyada weight
)

df_imputed = pd.DataFrame(
    knn_imputer.fit_transform(df_nums),
    columns=df_nums.columns
)

print("\nAfter KNN Imputation (k=3):")
print(df_imputed.round(2))

# ─── Before vs After — specific row check ───
print("\nAmit (row 2) — Rating was NaN:")
print(f"  Original: {df.loc[2, 'Rating']}")
print(f"  KNN fill: {df_imputed.loc[2, 'Rating']:.2f}")
# Amit ke 3 nearest neighbors (similar deliveries, exp, distance) ki rating use ki!

Original Data (numerical only):
   Deliveries_Day  Exp_Years  Avg_Distance_km  Rating
0            18.0        3.0              8.5     4.2
1            25.0        5.0             12.0     4.7
2             NaN        4.0             10.5     NaN
3            30.0        6.0              NaN     4.9
4            15.0        NaN              7.0     4.0
5            22.0        4.0              9.5     4.4
6            28.0        5.0              NaN     4.8
7             NaN        3.0              8.0     4.1
8            19.0        NaN              9.0     NaN
9            26.0        5.0             11.5     4.6

Missing values:
 Deliveries_Day     2
Exp_Years          2
Avg_Distance_km    2
Rating             2
dtype: int64

After KNN Imputation (k=3):
   Deliveries_Day  Exp_Years  Avg_Distance_km  Rating
0           18.00       3.00             8.50    4.20
1           25.00       5.00            12.00    4.70
2           24.93       4.00            10.50    4.58
3           30

### SimpleImputer vs KNN Imputer

In [6]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer

# ─── Same riders dataset ───
data = {
    'Deliveries_Day':  [18, 25, np.nan, 30, 15, 22, 28, np.nan, 19, 26],
    'Exp_Years':       [3,  5,  4,      6,  np.nan, 4, 5,  3,      np.nan, 5],
    'Rating':          [4.2, 4.7, np.nan, 4.9, 4.0, 4.4, 4.8, 4.1, np.nan, 4.6],
}
df = pd.DataFrame(data)

# ─── Method 1: SimpleImputer (mean) ───
mean_imputer = SimpleImputer(strategy='mean')
df_mean = pd.DataFrame(
    mean_imputer.fit_transform(df),
    columns=df.columns
)

# ─── Method 2: KNNImputer ───
knn_imputer = KNNImputer(n_neighbors=3)
df_knn = pd.DataFrame(
    knn_imputer.fit_transform(df),
    columns=df.columns
)

# ─── Compare results ───
print("Row 2 (Amit — Deliveries missing, Rating missing):")
print(f"  Original:        Deliveries=NaN,  Rating=NaN")
print(f"  Mean Imputed:    Deliveries={df_mean.loc[2,'Deliveries_Day']:.1f}, Rating={df_mean.loc[2,'Rating']:.2f}")
print(f"  KNN Imputed:     Deliveries={df_knn.loc[2,'Deliveries_Day']:.1f},  Rating={df_knn.loc[2,'Rating']:.2f}")

print("\nRow 4 (Pooja — Exp_Years missing):")
print(f"  Mean Imputed:    Exp_Years={df_mean.loc[4,'Exp_Years']:.2f}")
print(f"  KNN Imputed:     Exp_Years={df_knn.loc[4,'Exp_Years']:.2f}")
# KNN: Pooja ki Deliveries (15) aur Rating (4.0) jaisi rows dhundi
# → unka Exp_Years (2-3 years likely) use kiya
# Mean: sab ka average (avg~4.3 years) — less accurate for Pooja!

# 📊 KNN vs Mean: Junior riders (low deliveries, low rating) ke liye
# KNN kam experience deta hai — mean imputation sab ko global average de deta hai.
# KNN context-aware hai!


Row 2 (Amit — Deliveries missing, Rating missing):
  Original:        Deliveries=NaN,  Rating=NaN
  Mean Imputed:    Deliveries=22.9, Rating=4.46
  KNN Imputed:     Deliveries=25.3,  Rating=4.43

Row 4 (Pooja — Exp_Years missing):
  Mean Imputed:    Exp_Years=4.38
  KNN Imputed:     Exp_Years=3.33
